# R7 - Delta internals + optimisation

Proof that the lakehouse is auditable (Delta write history), restorable (time travel) and organised for the platform team's most common queries (month + hospital) without full rewrites.

Note: Delta-level history/time travel live on the pipeline's streaming tables (`claims.bronze.claims`, `claims.silver.policy_history`).

## 1. Audit trail — the table's own write history

Every pipeline update appends a version to the Delta log. `DESCRIBE HISTORY` is the proof the platform team asked for: who wrote, when, and how. Shown on the silver SCD2 streaming table and the bronze raw-claims streaming table.

In [0]:
%sql
describe history claims.silver.policy_history

In [0]:
%sql
describe history claims.bronze.claims

## 2. Time travel — pull an earlier version on demand

`VERSION AS OF` restores any snapshot in the history above. Verified readings: `policy_history` v2/v6/v14 all hold the 651 SCD2 versions; `bronze.claims` v4 already holds the full 3,040-claim ingest including CLM-30025. (Setup/property-only versions read as empty snapshots — e.g. policy_history v5, bronze v1/v10.)

A claim's numbers disputed weeks later can be pulled exactly as a given version recorded them:

In [0]:
%sql
select
  'policy_history current' as snapshot, count(*) as rows
from claims.silver.policy_history
union all
select
  'policy_history version 2', count(*)
from claims.silver.policy_history version as of 2
union all
select
  'bronze.claims version 4', count(*)
from claims.bronze.claims version as of 4
union all
select
  'bronze.claims current', count(*)
from claims.bronze.claims

In [0]:
%sql
-- a single disputed claim, exactly as version 4 recorded it
select claim_id, member_id, policy_id, billed_amount, approved_amount, room_days, status
from claims.bronze.claims version as of 4
where claim_id = 'CLM-30025'

## 3. File organisation — clustered by (admission_month, hospital_id)

Liquid clustering co-locates rows for the platform team's dominant access path: filter by month and/or hospital. New batches are organised incrementally — no full rewrite of existing data per batch.

Proof the layout is declared on the gold table:

In [0]:
%sql
show create table claims.gold.claims_enriched

## 4. Incremental writes — new batches never rewrite the table

The history itself is the proof: each pipeline update lands as a `STREAMING UPDATE` (bronze) / `MERGE` (silver) commit on top of the existing data — no full-table rewrite per batch. The gold MV refreshes incrementally for the same reason.

In [0]:
%sql
-- bronze: every batch appends as a STREAMING UPDATE commit; only setup/property snapshots in between
select version, operation
from (describe history claims.bronze.claims)
order by version

In [0]:
%sql
-- silver SCD2: each feed batch is an incremental MERGE, not a rewrite
select version, timestamp, operation
from (describe history claims.silver.policy_history)
where operation in ('MERGE', 'DLT SETUP')
order by version